# july24_pnp dataset visualization

Visualizes episodes from the LeRobot v3.0 dataset at `~/checkpoints/maniflow/july24_pnp/`
(512 episodes, Franka pick-and-place, 25 fps, 3 RGB + 3 depth cameras).

**Units and normalization notes** (verified against `meta/stats.json`):

- Per-key features (`observation.state.joint_position`, `action.joint_target`, ...) are stored
  **unnormalized**, in radians. Frame 0 of every episode equals `q_home`.
- The composite `action` vector (what a policy typically trains on) stores **relative joint deltas**
  in its first 7 dims (identical to `action.relative_joint_target`), plus binary gripper and a
  `hard_config` flag. To get physically meaningful joint actions we *unnormalize* them back to
  absolute joint space: the absolute target is `action.joint_target`
  (≈ `observation.state.joint_position + action[:7]`, the small residual being due to the delta
  being relative to the previous *target* rather than the measured state).
- Video files map 1:1 to episodes (`file-{episode:03d}.mp4`); 42 of 512 episodes have no video.

In [ ]:
import base64
import json
from pathlib import Path

import av
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

# DATASET_ROOT = Path.home() / "dataset/july24_pnp"
# DATASET_ROOT = Path.home() / "dataset/aug11_flip/blocks_success"

DATASET_ROOT = Path.home() / "datasets/aug13_sim2real_kp40_pnp/sim2realpnp/blocks_success"
# DATASET_ROOT = Path.home() / "dataset/compliant_test_flip_pnp_softer/blocks_success"
# DATASET_ROOT = Path.home() / "checkpoints/maniflow/pnp_stiff/blocks_success"
# DATASET_ROOT = Path("/home/tobi/code/blocksuite/datasets/july27_pnp_stiff_kp100/blocks_success")

info = json.loads((DATASET_ROOT / "meta/info.json").read_text())
stats = json.loads((DATASET_ROOT / "meta/stats.json").read_text())
FPS = info["fps"]

Q_HOME = np.array([0.0, -np.pi / 4, 0.0, -3 * np.pi / 4, 0.0, np.pi / 2, 0.0])

RGB_CAMERAS = [
    "observation.images.camera:front_camera",
    "observation.images.camera:left_camera",
    "observation.images.camera:gripper_camera",
]

# Episode-level metadata. In LeRobot v3 several episodes can share one data
# parquet, so we also keep the data/* mapping columns (which file holds each
# episode and its global row range) to load a single episode correctly.
_meta_cols = [
    "episode_index",
    "length",
    "tasks",
    "data/chunk_index",
    "data/file_index",
    "dataset_from_index",
    "dataset_to_index",
]
episodes_meta = pd.concat(
    [
        pd.read_parquet(f, columns=_meta_cols)
        for f in sorted((DATASET_ROOT / "meta/episodes/chunk-000").glob("*.parquet"))
    ],
    ignore_index=True,
).set_index("episode_index")

print(f"{len(episodes_meta)} episodes, {episodes_meta['length'].sum()} frames total, {FPS} fps")
episodes_meta.head()

In [ ]:
def load_episode(episode_index: int) -> pd.DataFrame:
    """Load all frames of one episode.

    LeRobot v3 may pack several episodes into a single data parquet, so we open
    the file that holds this episode (from the meta mapping) and keep only its
    rows. Filtering on ``episode_index`` is robust to row ordering within a file.
    """
    row = episodes_meta.loc[episode_index]
    chunk = int(row["data/chunk_index"])
    file = int(row["data/file_index"])
    df = pd.read_parquet(DATASET_ROOT / f"data/chunk-{chunk:03d}/file-{file:03d}.parquet")
    return df[df["episode_index"] == episode_index].reset_index(drop=True)


def video_path(camera_key: str, episode_index: int) -> Path | None:
    p = DATASET_ROOT / f"videos/{camera_key}/chunk-000/file-{episode_index:03d}.mp4"
    return p if p.exists() else None


def stack_col(df: pd.DataFrame, col: str) -> np.ndarray:
    return np.stack(df[col].to_numpy())


# Sanity check: dataset stores physical (unnormalized) values.
# Frame 0 joint positions of episode 0 must equal q_home, and the stats.json
# ranges for joint positions must be plausible radians (within Franka limits).
ep0 = load_episode(0)
q0 = stack_col(ep0, "observation.state.joint_position")[0]
# assert np.allclose(q0, Q_HOME, atol=1e-5), "expected raw radians in the parquet files"
print("frame 0 of episode 0 == q_home  ->  values are stored unnormalized (radians)")
print("joint_position range from stats.json:")
print("  min", np.round(stats["observation.state.joint_position"]["min"], 3))
print("  max", np.round(stats["observation.state.joint_position"]["max"], 3))

## Pick an episode

Change `EPISODE` and re-run the cells below. Episodes without video: see `missing_video`.

In [ ]:
missing_video = sorted(
    ep for ep in episodes_meta.index if video_path(RGB_CAMERAS[0], ep) is None
)
print(f"{len(missing_video)} episodes without video, e.g. {missing_video[:10]}")

EPISODE = 17

ep = load_episode(EPISODE)
t = stack_col(ep, "timestamp").squeeze()
n_frames = len(ep)
# Note: is_episode_successful is False for all 512 episodes (not populated), so it is omitted here.
print(
    f"episode {EPISODE}: {n_frames} frames ({n_frames / FPS:.1f} s), "
    f"task={episodes_meta.loc[EPISODE, 'tasks'][0]}"
)

## Camera videos

Embedded mp4 players for the three RGB cameras (AV1-encoded; plays in Chromium-based frontends).
The following cell also renders a static frame strip per camera in case the player doesn't work.

In [ ]:
def embed_videos(episode_index: int, camera_keys: list[str]) -> HTML:
    tags = []
    for key in camera_keys:
        name = key.split("camera:")[-1]
        path = video_path(key, episode_index)
        if path is None:
            tags.append(f"<div style='width:32%'><b>{name}</b><br>no video</div>")
            continue
        b64 = base64.b64encode(path.read_bytes()).decode()
        tags.append(
            f"<div style='width:32%'><b>{name}</b><br>"
            f"<video controls loop muted style='width:100%'>"
            f"<source src='data:video/mp4;base64,{b64}' type='video/mp4'></video></div>"
        )
    return HTML(f"<div style='display:flex;gap:1%'>{''.join(tags)}</div>")


display(embed_videos(EPISODE, RGB_CAMERAS))

In [ ]:
def decode_frames(path: Path, indices: list[int]) -> list[np.ndarray]:
    """Decode selected frame indices from a video (sequential decode, AV1-safe)."""
    frames, wanted = [], set(indices)
    with av.open(str(path)) as container:
        for i, frame in enumerate(container.decode(video=0)):
            if i in wanted:
                frames.append(frame.to_ndarray(format="rgb24"))
            if i >= max(wanted):
                break
    return frames


N_STRIP = 6
strip_idx = np.linspace(0, n_frames - 1, N_STRIP).astype(int).tolist()

fig, axes = plt.subplots(len(RGB_CAMERAS), N_STRIP, figsize=(3 * N_STRIP, 5.5))
for row, key in enumerate(RGB_CAMERAS):
    name = key.split("camera:")[-1]
    path = video_path(key, EPISODE)
    frames = decode_frames(path, strip_idx) if path else []
    for col in range(N_STRIP):
        ax = axes[row, col]
        ax.axis("off")
        if col < len(frames):
            ax.imshow(frames[col])
        if row == 0:
            ax.set_title(f"t = {strip_idx[col] / FPS:.1f} s", fontsize=10)
        if col == 0:
            ax.text(-0.05, 0.5, name, transform=ax.transAxes, rotation=90,
                    va="center", ha="right", fontsize=11)
fig.suptitle(f"Episode {EPISODE} frame strip", y=1.0)
fig.tight_layout()
plt.show()

## Joint observations and actions (unnormalized, radians)

Per joint: measured position (`observation.state.joint_position`) and the absolute action.
The absolute action is the stored relative delta `action[:7]` unnormalized back to joint space,
which equals `action.joint_target`. The dashed grey line is `q_home`.

In [ ]:
q_obs = stack_col(ep, "observation.state.joint_position")          # (T, 7) radians
action = stack_col(ep, "action")                                    # (T, 9) rel deltas + gripper + hard_config
q_action = q_obs + action[:, :7]                                    # unnormalized: relative delta -> absolute
q_target = stack_col(ep, "action.joint_target")                     # absolute target stored in dataset

# The unnormalized action should match the stored absolute target up to the
# target-vs-measured-state residual.
print(f"max |(state + action) - joint_target| = {np.abs(q_action - q_target).max():.4f} rad")

gripper_obs = stack_col(ep, "observation.state.gripper_position").squeeze()
gripper_action = action[:, 7]

# pd_mode is the last action dimension (0/1). It is also in observation.state
# when that vector is 9-dim (7 joints + gripper + pd_mode); older data is 8-dim.
pd_mode_action = action[:, 8]
_obs_state = stack_col(ep, "observation.state")
pd_mode_obs = _obs_state[:, 8] if _obs_state.shape[1] >= 9 else None

fig, axes = plt.subplots(5, 2, figsize=(13, 15), sharex=True)
axes = axes.ravel()
for j in range(7):
    ax = axes[j]
    ax.plot(t, q_obs[:, j], label="observed position", color="tab:blue", lw=1.5)
    ax.plot(t, q_target[:, j], label="action (absolute target)", color="tab:orange", lw=1.0, ls="--")
    ax.axhline(Q_HOME[j], color="grey", ls=":", lw=1.5, label="q_home")
    ax.set_title(f"joint_{j}")
    ax.set_ylabel("rad")
    ax.grid(alpha=0.3)
    if j == 0:
        ax.legend(loc="best", fontsize=8)

ax = axes[7]
ax.plot(t, gripper_obs, label="observed gripper", color="tab:blue", lw=1.5)
ax.plot(t, gripper_action, label="gripper action (binary)", color="tab:orange", lw=1.0, ls="--")
ax.set_title("gripper")
ax.grid(alpha=0.3)
ax.legend(loc="best", fontsize=8)

ax = axes[8]
if pd_mode_obs is not None:
    ax.plot(t, pd_mode_obs, label="observed pd_mode", color="tab:blue", lw=1.5)
ax.plot(t, pd_mode_action, label="pd_mode action (binary)", color="tab:orange", lw=1.0, ls="--")
ax.set_title("pd_mode")
ax.set_ylim(-0.1, 1.1)
ax.grid(alpha=0.3)
ax.legend(loc="best", fontsize=8)

axes[9].set_visible(False)

for ax in (axes[7], axes[8]):
    ax.set_xlabel("time [s]")
fig.suptitle(f"Episode {EPISODE}: joint observations vs. actions (unnormalized)", y=1.0)
fig.tight_layout()
plt.show()

## Raw relative action deltas

What the policy actually sees as `action[:7]` (relative joint targets per step, radians).

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
for j in range(7):
    ax.plot(t, action[:, j], label=f"joint_{j}", lw=1.0)
ax.axhline(0.0, color="grey", ls=":", lw=1.0)
ax.set_xlabel("time [s]")
ax.set_ylabel("delta [rad]")
ax.set_title(f"Episode {EPISODE}: relative joint action deltas (action[:7])")
ax.grid(alpha=0.3)
ax.legend(ncol=7, fontsize=8, loc="upper right")
plt.show()

## 3D end-effector trajectory (interactive)

Plotly figure — rotate/zoom with the mouse.

- **blue-to-yellow line**: dataset `observation.state.ee_position[:3]` (flange pose, meters, robot base frame), colored by time.
- **red line**: forward kinematics of the joint observations computed with the `rcs` kinematics
  module (`rcs.common.Pin`, FR3 model, DROID setup with Robotiq 2F-85 TCP offset). This is the
  actual TCP between the fingers; the dataset `ee_position` has no gripper offset, so the FK line
  runs a constant 0.1493 m (the 2F-85 TCP z-offset) from the dataset line.
  Set `SHOW_FK = False` to hide it.
- Markers: start/end, gripper close/open events, block positions at frame 0.

In [ ]:
# import plotly.graph_objects as go
# import rcs
# from rcs import common as rcs_common

# SHOW_FK = True  # set to False to hide the rcs forward-kinematics line

# ee = stack_col(ep, "observation.state.ee_position")  # (T, 6): x, y, z, roll, pitch, yaw
# xyz = ee[:, :3]

# # Forward kinematics of the joint observations: FR3 + Robotiq 2F-85 TCP (DROID setup)
# _robot = rcs.ROBOTS[rcs_common.RobotType.FR3]
# _pin = rcs_common.Pin(_robot.mjcf_model_path, _robot.attachment_site, False)
# _tcp_2f85 = rcs.GRIPPER_TCP_OFFSETS[rcs_common.GripperType("Robotiq2F85")]
# fk_xyz = np.array([_pin.forward(qi, _tcp_2f85).translation() for qi in q_obs])

# # Frames where the binary gripper action toggles
# toggles = np.flatnonzero(np.diff(gripper_action) != 0) + 1

# fig = go.Figure()

# fig.add_trace(go.Scatter3d(
#     x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
#     mode="lines+markers",
#     marker=dict(size=2.5, color=t, colorscale="Viridis",
#                 colorbar=dict(title="time [s]", len=0.6)),
#     line=dict(color="grey", width=2),
#     name="dataset ee_position (flange)",
#     hovertemplate="t=%{text:.2f}s<br>x=%{x:.3f} y=%{y:.3f} z=%{z:.3f}<extra></extra>",
#     text=t,
# ))

# if SHOW_FK:
#     fig.add_trace(go.Scatter3d(
#         x=fk_xyz[:, 0], y=fk_xyz[:, 1], z=fk_xyz[:, 2],
#         mode="lines",
#         line=dict(color="crimson", width=4),
#         name="rcs FK (2F-85 TCP)",
#         hovertemplate="t=%{text:.2f}s<br>x=%{x:.3f} y=%{y:.3f} z=%{z:.3f}<extra></extra>",
#         text=t,
#     ))

# fig.add_trace(go.Scatter3d(
#     x=[xyz[0, 0]], y=[xyz[0, 1]], z=[xyz[0, 2]], mode="markers",
#     marker=dict(size=8, color="green", symbol="circle"), name="start"))
# fig.add_trace(go.Scatter3d(
#     x=[xyz[-1, 0]], y=[xyz[-1, 1]], z=[xyz[-1, 2]], mode="markers",
#     marker=dict(size=8, color="red", symbol="square"), name="end"))

# close_idx = [k for k in toggles if gripper_action[k] > gripper_action[k - 1]]
# open_idx = [k for k in toggles if gripper_action[k] < gripper_action[k - 1]]
# for idx, name, symbol in [(close_idx, "gripper close", "x"), (open_idx, "gripper open", "diamond")]:
#     if idx:
#         fig.add_trace(go.Scatter3d(
#             x=xyz[idx, 0], y=xyz[idx, 1], z=xyz[idx, 2], mode="markers",
#             marker=dict(size=6, color="black", symbol=symbol), name=name))

# _obj_prefix = "observation.state.object_pose."
# _obj_cols = [c for c in ep.columns if c.startswith(_obj_prefix) and c != _obj_prefix + "workspace_table"]
# _obj_palette = ["orange", "purple", "cyan", "magenta", "brown", "olive", "teal", "pink"]
# for col, color in zip(_obj_cols, _obj_palette):
#     obj = col[len(_obj_prefix):]
#     pose = stack_col(ep, col)[0]
#     fig.add_trace(go.Scatter3d(
#         x=[pose[0]], y=[pose[1]], z=[pose[2]], mode="markers",
#         marker=dict(size=7, color=color, symbol="diamond"), name=f"{obj} (t=0)"))

# fig.update_layout(
#     title=f"Episode {EPISODE}: end-effector trajectory",
#     scene=dict(
#         xaxis_title="x [m]", yaxis_title="y [m]", zaxis_title="z [m]",
#         aspectmode="data",
#     ),
#     height=750,
#     legend=dict(x=0.02, y=0.98),
#     margin=dict(l=0, r=0, t=40, b=0),
# )
# fig.show()

## Execute the episode's joint actions on the robot

Based on `examples/fr3/fr3_env_joint.py`: both the simulation and the hardware env are built
through the config pattern (`scene.config()` / `env_creator.config()` + `create_env(cfg)`).

- **Simulation**: `EmptyWorldFR3` scene (MuJoCo GUI opens on env creation).
- **Hardware**: `DROIDEnv` from `rcs_fr3.configs` — single FR3 + Robotiq 2F-85, the same
  DROID setup this dataset was recorded with.

The actions sent are the episode's absolute joint targets (`action.joint_target`, radians,
already unnormalized) plus the binary gripper command, stepped at the dataset frame rate.

**End-effector load / gravity compensation**: the FR3 only compensates gravity for the payload
it has been told about. `FRANKA_LOAD` below describes the Robotiq 2F-85 + wrist ZED so the
controller's gravity term is right; without it `tau_J` carries an uncompensated ~1 kg and the soft
PD mode (`kp = 40`) sags. `m_total = m_ee + m_load`, where `m_ee` comes from Desk's End Effector
slot and `m_load` from this config — set `DESK_EE_IS_ZEROED` to match your Desk setup.

**Safety**: the cell only moves a robot if you set `EXECUTE_ON_ROBOT = True`. With
`RobotPlatform.HARDWARE` it connects to `FR3_IP` and waits for Enter before moving.

In [ ]:
raise NotImplementedError("TODO: visualize object poses and gripper state in 3D")

In [ ]:
from pathlib import Path
from time import sleep
import typing

from rcs._core.common import BaseCameraConfig, RobotPlatform
from rcs._core.sim import SimConfig
from rcs.envs.base import ControlMode, LimitedAbsoluteAction, RelativeTo, SimpleFrameRate
from rcs.envs.configs import EmptyWorldFR3
from rcs_fr3.creators import HardwareCameraCreatorConfig

import rcs
import rcs_fr3


ROBOT_INSTANCE = RobotPlatform.HARDWARE  # or RobotPlatform.SIMULATION
FR3_IP = "192.168.1.12"
ROBOTIQ_SERIAL = "DAANTG8W"
ROBOT_NAME = "right"  # lead robot name in both EmptyWorldFR3 and DROIDEnv

# ZED cameras (same setup as examples/teleop/franka.py). These are recorded
# during the replay and shown alongside the joint plot in the last cell.
ZED_CAMERA_DICT = {
    "gripper_rgb": "14943057",
    "front_rgb": "35115330",
}
ZED_CAMERA_DICT = None
INCLUDE_DEPTH = False

HOME_POSE = rcs.HOME_POSITIONS["FR3_DROID"]

# --- End-effector load (gravity compensation) --------------------------------
# libfranka keeps two additive payload slots: the End Effector configured in
# Desk (m_ee / F_x_Cee) and the load set here via setLoad (m_load / F_x_Cload),
# with m_total = m_ee + m_load. Both centres of mass are expressed in the
# FLANGE frame, in metres.
#
#   DESK_EE_IS_ZEROED = True  -> Desk's EE slot is empty/zeroed, so the whole
#                                stack (2F-85 + coupler + camera) goes here.
#   DESK_EE_IS_ZEROED = False -> Desk already describes the 2F-85 + coupler, so
#                                only the camera assembly goes here.
# Check Desk before trusting this: a DROID cell often still has the *Franka
# Hand* (0.73 kg) configured there, which silently biases m_total.
# Set to False after 1.165 kg made the arm push itself up and lock: that extra
# gravity term is only right if Desk's EE slot is really empty. It is not --
# Desk already carries the 2F-85 + coupler, so only the camera goes below.
DESK_EE_IS_ZEROED = False

# Robotiq 2F-85 (0.925 kg datasheet) + Franka<->Robotiq coupler plate. The 2F-85
# is mounted with a +90 deg rotation about flange z (rcs.GRIPPER_MOUNT_OFFSETS),
# but it is near-symmetric about that axis, so its CoM sits essentially on the
# flange z axis and the rotation does not change C_GRIPPER.
M_GRIPPER = 0.925 + 0.15
C_GRIPPER = np.array([0.0, 0.0, 0.058])

# Wrist ZED Mini (63 g) + bracket + the hanging weight of the USB-C cable.
# Weigh the assembly -- the bracket and cable are easily half of this.
M_CAMERA = 0.09 #5

# Bounding-box extents (x, y, z in the flange frame, metres) used to turn each
# mass into an inertia tensor. The FR3's control unit validates the tensor it is
# handed: a zero matrix with mass > 0 is not a physically valid rigid body and
# setLoad comes back as "Set Load command rejected: invalid argument!". These
# only have to be plausible -- the inertial term is small next to gravity at
# this speed -- but they must describe a real body.
GRIPPER_EXTENTS = np.array([0.09, 0.15, 0.15])  # 2F-85 + coupler plate
CAMERA_EXTENTS = np.array([0.03, 0.125, 0.03])  # ZED Mini + bracket
# Fallback CoM used when the hand-eye calibration file is missing. The camera is
# the off-axis mass: it is what makes the gravity error rotate with wrist yaw.
C_CAMERA_FALLBACK = np.array([0.0, -0.045, 0.09])

# Hand-eye result written by camera_calibration.py (same directory).
CALIB_NPZ = Path("camera_calibration_result.npz")


def _camera_com_in_flange() -> np.ndarray:
    """Camera CoM in the flange frame, from the hand-eye calibration if present.

    ``T_ee_cam_wrist`` maps camera -> TCP, and the 2F-85 TCP is a pure
    translation of +0.1493 m along flange z, so shifting the translation by that
    offset lands in the flange frame. This is the optical centre rather than the
    centre of mass, but for a ZED Mini the two are within a couple of cm and it
    beats guessing the lever arm.
    """
    if not CALIB_NPZ.exists():
        print(f"{CALIB_NPZ} not found, using C_CAMERA_FALLBACK")
        return C_CAMERA_FALLBACK
    T_ee_cam = np.asarray(np.load(CALIB_NPZ, allow_pickle=True)["T_ee_cam_wrist"])
    tcp = rcs.GRIPPER_TCP_OFFSETS[rcs._core.common.GripperType("Robotiq2F85")]
    return T_ee_cam[:3, 3] + tcp.translation()


def _box_inertia(mass: float, extents: np.ndarray) -> np.ndarray:
    """Inertia of a homogeneous box about its own centre of mass."""
    x2, y2, z2 = np.asarray(extents, dtype=float) ** 2
    return np.diag(mass / 12.0 * np.array([y2 + z2, x2 + z2, x2 + y2]))


def _shift_inertia(inertia: np.ndarray, mass: float, d: np.ndarray) -> np.ndarray:
    """Parallel-axis shift of an about-CoM tensor by the offset ``d``."""
    d = np.asarray(d, dtype=float)
    return inertia + mass * (float(d @ d) * np.eye(3) - np.outer(d, d))


def make_franka_load() -> "rcs_fr3.hw.FrankaLoad":
    """Compose the payload the FR3 should compensate for."""
    m_cam, c_cam = M_CAMERA, _camera_com_in_flange()
    if DESK_EE_IS_ZEROED:
        mass = M_GRIPPER + m_cam
        com = (M_GRIPPER * C_GRIPPER + m_cam * c_cam) / mass
        # setLoad wants the tensor about the *load's* CoM (like the documented
        # Franka Hand values), so shift both boxes onto the combined CoM.
        inertia = _shift_inertia(
            _box_inertia(M_GRIPPER, GRIPPER_EXTENTS), M_GRIPPER, C_GRIPPER - com
        ) + _shift_inertia(_box_inertia(m_cam, CAMERA_EXTENTS), m_cam, c_cam - com)
    else:
        mass, com = m_cam, c_cam
        inertia = _box_inertia(m_cam, CAMERA_EXTENTS)
    load = rcs_fr3.hw.FrankaLoad()
    load.load_mass = float(mass)
    load.f_x_cload = com
    load.load_inertia = inertia
    eig = np.linalg.eigvalsh(inertia)
    assert eig.min() > 0.0, f"load inertia is not positive definite: {eig}"
    print(f"load: mass={mass:.3f} kg, com={np.round(com, 4)} m (flange frame)")
    print(f"      inertia diag={np.round(np.diag(inertia), 6)} kg m^2, eig min={eig.min():.2e}")
    return load


# Payload declaration disabled -- uncomment both this line and the
# load_parameters assignment in get_env() to put it back.
# FRANKA_LOAD = make_franka_load()

# Actions from the loaded episode: absolute joint targets (rad, unnormalized)
# and binary gripper command.
robot_actions = np.concatenate(
    [
        stack_col(ep, "action.joint_target"),  # (T, 7) absolute joint targets
        1 - action[:, 7:8],                    # (T, 1) binary gripper (inverted)
        action[:, 8:9],                        # (T, 1) pd_mode
    ],
    axis=1,
)


def _zed_camera_cfgs():
    """ZED camera creator config mirroring examples/teleop/franka.py."""
    if not ZED_CAMERA_DICT:
        return None
    return {
        "zed": HardwareCameraCreatorConfig(
            camera_type_id="zed",
            camera_cfgs={
                name: BaseCameraConfig(
                    identifier=identifier,
                    resolution_width=1280,
                    resolution_height=720,
                    frame_rate=30,
                )
                for name, identifier in ZED_CAMERA_DICT.items()
            },
            kwargs={
                "enable_depth": INCLUDE_DEPTH,
                "enable_imu": False,
                "include_right": not INCLUDE_DEPTH,
            },
        )
    }


def get_env():
    if ROBOT_INSTANCE == RobotPlatform.HARDWARE:
        from rcs_fr3.configs import DROIDEnv

        env_creator = DROIDEnv()
        env_creator.robot_ip = FR3_IP
        env_creator.gripper_serial_number = ROBOTIQ_SERIAL
        hw_cfg = env_creator.config()
        hw_cfg.control_mode = ControlMode.JOINTS
        hw_cfg.max_relative_movement = np.deg2rad(100)
        hw_cfg.relative_to = RelativeTo.NONE
        hw_cfg.wrapper_cfg.binary_gripper = True
        hw_cfg.wrapper_cfg.include_depth = INCLUDE_DEPTH
        hw_cfg.camera_cfgs = _zed_camera_cfgs()
        hw_cfg.robot_cfgs[ROBOT_NAME].q_home = HOME_POSE
        hw_cfg.robot_cfgs[ROBOT_NAME].ignore_realtime = True
        hw_cfg.robot_cfgs[ROBOT_NAME].speed_factor = 0.5
        # hw_cfg.robot_cfgs[ROBOT_NAME].load_parameters = FRANKA_LOAD
        # On each (re)start of the joint controller (incl. after a PD-mode
        # switch) the controller holds the current position and ramps its target
        # to the commanded action over a gap-scaled window, blocking until
        # converged, so the stiff controller resumes with ~zero error instead of
        # jumping. approach_joint_speed sets the max joint speed of that ramp.
        hw_cfg.robot_cfgs[ROBOT_NAME].blocking_move_on_start = False
        hw_cfg.robot_cfgs[ROBOT_NAME].approach_joint_speed = 0.4
        hw_cfg.gripper_cfgs[ROBOT_NAME].async_control = True
        hw_cfg.gripper_cfgs[ROBOT_NAME].policy_rate = int(FPS)
        env_rel = env_creator.create_env(hw_cfg)
        # input("the robot is going to move, press enter whenever you are ready")
    else:
        scene = EmptyWorldFR3()
        sim_cfg_data = scene.config()
        sim_cfg_data.sim_cfg = SimConfig(
            async_control=True, realtime=True, frequency=FPS, max_convergence_steps=500
        )
        sim_cfg_data.control_mode = ControlMode.JOINTS
        sim_cfg_data.relative_to = RelativeTo.NONE
        sim_cfg_data.robot_cfgs[scene.lead_robot_name(sim_cfg_data)].q_home = HOME_POSE
        env_rel = scene.create_env(sim_cfg_data)

    return env_rel


def set_pd_mode(mode: float, cfg: rcs_fr3.hw.FrankaConfig):
    # Stream set-points at the dataset frame rate so the controller's
    # interpolation window (1 / policy_rate) matches the replay step period.
    cfg.policy_rate = int(FPS)
    if mode == 1.0:
        # cfg.kd = np.ones(7) * 80
        # cfg.kd = np.array([20., 20., 20., 20., 7.5, 15.0, 5.0])
        # cfg.kp = np.array([100., 100., 100., 100., 75., 150., 50.])
                # cfg.kp = np.ones(7) * 880
#          common::Vector7d kp =
#       (common::Vector7d() << 100., 100., 100., 100., 75., 150., 50.).finished();
#   common::Vector7d kd =
#       (common::Vector7d() << 20., 20., 20., 20., 7.5, 15.0, 5.0).finished();
        cfg.kp = np.array([600, 600, 600, 600, 250, 150, 50])
        cfg.kd = np.array([50, 50, 50, 50, 30, 25, 15])
        # cfg.kp = np.array([80., 80., 80., 80., 80, 80, 50.])
        cfg.torque_limit = np.array([87.0, 87.0, 87.0, 87.0, 87.0, 87.0, 87.0])  # Franka torque limits
        cfg.blocking_move_on_start = True # soft to stiff: we want to sync
        assert False
    elif mode == 0.0:
        # raise ValueError("PD mode 0.0 is not supported on the real robot (too low gains)")
        # cfg.kd = np.ones(7) * 18
        # cfg.kp = np.ones(7) * 80
        # raise ValueError("PD mode 0.0 is not supported on the real robot (too low gains)")
        cfg.kp = np.array([40., 40., 40., 40., 40., 40., 40.])
        cfg.kd = np.array([13., 13., 13., 13., 13., 13., 13.0])
        cfg.torque_limit = np.array([87.0, 87.0, 87.0, 87.0, 12.0, 12.0, 12.0])  # Franka torque limits
        cfg.blocking_move_on_start = False # stiff to soft, we dont want to sync
    else:
        raise ValueError(f"Invalid pd mode: {mode}")
    return cfg


def run_episode_on_robot(actions: np.ndarray):
    """Replay absolute joint actions and return recorded joint observations.

    Returns
    -------
    q_robot : (T, 7) joint positions read from the robot after each step
    gripper_robot : (T,) gripper readings after each step
    q_sent : (T, 7) absolute joint target the env actually handed to the robot,
        i.e. after `LimitedAbsoluteAction` clipped the commanded target to
        +/- `max_relative_movement` around the *measured* position and to the
        joint limits. Read back from `info[robot]["limited_absolute_action"]`.
    gripper_sent : (T,) gripper value passed into `env.step()` (already inverted
        relative to the dataset's `action[:, 7]`)
    tau_induced : (T, 7) torque the joint controller commanded, read from
        `FrankaState.tau_J_d`. This is the PD law of `Franka.cpp:705`
        `Kp * (desired_q - q) - Kd * dq`, after the joint-limit cutoff and the
        `torque_limit` clamp -- i.e. exactly what `set_pd_mode` controls.
        Excludes gravity (libfranka adds it on the robot side).
    tau_measured : (T, 7) link-side joint torque actually measured by the robot
        (`FrankaState.tau_J`). Includes gravity, so it is NOT directly
        comparable in absolute terms to `tau_induced`.
    camera_frames : dict[str, list[np.ndarray]] recorded ZED RGB frames per camera
    replay_error : Exception | None
    """
    q_robot = []
    gripper_robot = []
    q_sent = []
    gripper_sent = []
    tau_induced = []
    tau_measured = []
    torque_error = None
    camera_frames: dict[str, list[np.ndarray]] = {}
    replay_error = None
    env_rel = None

    def record_frames(obs):
        # CameraSetWrapper stores frames under obs["frames"][cam]["rgb"]["data"].
        for cam_name, cam in obs.get("frames", {}).items():
            rgb = cam.get("rgb")
            if rgb is None:
                continue
            camera_frames.setdefault(cam_name, []).append(np.asarray(rgb["data"]))

    try:
        frame_rate = SimpleFrameRate(FPS)
        env_rel = get_env()
        robot = typing.cast(rcs_fr3.hw.Franka, env_rel.get_wrapper_attr("robot")["right"])
        cfg = robot.get_config()
        prev_mode = actions[0][8]
        robot.set_config(set_pd_mode(actions[0][8], cfg))
        obs, info = env_rel.reset()

        for act in actions[:-1]:
            print(act[8])
            act_dict = {ROBOT_NAME: {"joints": act[:7], "gripper": [act[7]]}}
            if act[8] != prev_mode:
                robot.stop_control_thread()
                robot.set_config(set_pd_mode(act[8], cfg))
                prev_mode = act[8]
                # sleep(1.0)

            obs, reward, terminated, truncated, info = env_rel.step(act_dict)
            q_robot.append(np.asarray(obs[ROBOT_NAME]["joints"], dtype=np.float64))
            gripper_robot.append(float(np.asarray(obs[ROBOT_NAME]["gripper"]).ravel()[0]))
            # What the env really commanded: LimitedAbsoluteAction publishes its
            # post-clipping absolute target in the per-robot info dict. Falls back
            # to the requested action when that wrapper is not in the stack
            # (i.e. relative_to != RelativeTo.NONE or max_relative_movement=None).
            _sent = info.get(ROBOT_NAME, {}).get(LimitedAbsoluteAction.ABSOLUTE_ACTION_KEY)
            q_sent.append(
                np.asarray(act[:7] if _sent is None else _sent, dtype=np.float64)[:7]
            )
            gripper_sent.append(float(act[7]))
            # Commanded (tau_J_d) and measured (tau_J) joint torques. Only the
            # hardware Franka exposes these; NaN keeps the arrays aligned if not.
            try:
                _st = robot.get_state()
                tau_induced.append(np.asarray(_st.tau_J_d, dtype=np.float64))
                tau_measured.append(np.asarray(_st.tau_J, dtype=np.float64))
            except Exception as torque_exc:  # noqa: BLE001 - report, don't abort the replay
                if torque_error is None:
                    torque_error = torque_exc
                tau_induced.append(np.full(7, np.nan))
                tau_measured.append(np.full(7, np.nan))
            record_frames(obs)
            # frame_rate()
            sleep(1.0 / FPS)
    except Exception as exc:
        replay_error = exc
        print(f"Replay stopped after {len(q_robot)} samples: {type(exc).__name__}: {exc}")
    finally:
        if env_rel is not None:
            try:
                env_rel.close()
            except Exception as close_exc:
                if replay_error is None:
                    replay_error = close_exc
                    print(f"Replay cleanup failed: {type(close_exc).__name__}: {close_exc}")

    q_robot_arr = np.stack(q_robot) if q_robot else np.empty((0, 7), dtype=np.float64)
    q_sent_arr = np.stack(q_sent) if q_sent else np.empty((0, 7), dtype=np.float64)
    tau_ind_arr = np.stack(tau_induced) if tau_induced else np.empty((0, 7), dtype=np.float64)
    tau_meas_arr = np.stack(tau_measured) if tau_measured else np.empty((0, 7), dtype=np.float64)
    if torque_error is not None:
        print(
            f"WARNING: joint torques unavailable, all NaN "
            f"({type(torque_error).__name__}: {torque_error})"
        )
    return (
        q_robot_arr,
        np.asarray(gripper_robot),
        q_sent_arr,
        np.asarray(gripper_sent),
        tau_ind_arr,
        tau_meas_arr,
        camera_frames,
        replay_error,
    )


(
    q_robot,
    gripper_robot,
    q_sent,
    gripper_sent,
    tau_induced,
    tau_measured,
    camera_frames,
    replay_error,
) = run_episode_on_robot(robot_actions)
print(f"recorded {len(q_robot)} joint observations from {ROBOT_INSTANCE}")
# How often LimitedAbsoluteAction had to clip: sent target != commanded target.
_delta_sent = np.abs(q_sent - robot_actions[: len(q_sent), :7]).max(axis=1)
print(
    f"steps where the sent target differed from the commanded action: "
    f"{int((_delta_sent > 1e-9).sum())}/{len(q_sent)} "
    f"(max deviation {_delta_sent.max() if len(_delta_sent) else 0.0:.5f} rad)"
)
if len(tau_induced) and np.isfinite(tau_induced).any():
    _amp_ind = np.nanmax(np.abs(tau_induced), axis=0)
    _amp_meas = np.nanmax(np.abs(tau_measured), axis=0)
    print(f"induced |tau_J_d| max per joint = {np.round(_amp_ind, 2)} Nm")
    print(f"measured |tau_J|  max per joint = {np.round(_amp_meas, 2)} Nm")
    # tau_J_d carries no gravity term and is driven by kp/kd, so it is typically
    # far smaller than tau_J. On a shared y-axis it collapses onto zero, which is
    # why the torque panels give it its own (red) axis.
    print(f"ratio |tau_J_d| / |tau_J| per joint = {np.round(_amp_ind / np.maximum(_amp_meas, 1e-9), 3)}")
else:
    print("no joint torques recorded (all NaN) - see the WARNING above for the reason")
if camera_frames:
    print("recorded ZED frames: " + ", ".join(f"{name}={len(f)}" for name, f in camera_frames.items()))
if replay_error is not None:
    print("Comparison below contains the samples captured before the replay failure.")


## Dataset vs. robot joint comparison

For each joint: dataset observation (`observation.state.joint_position`), dataset absolute
action (`action.joint_target`), the target the env actually sent to the robot inside `step()`
(`info["limited_absolute_action"]`, i.e. after `LimitedAbsoluteAction` clipping — red), and the
joint reading recorded from the robot while replaying those actions. Dataset traces use their recorded timestamps; robot traces use an ideal `1 / FPS`
frame clock, so PD-mode reconfiguration delays do not shift later robot samples. Requires the
previous cell to have been run so `q_robot` exists.
Dashed grey = `q_home`.

The last three panels add end-effector **x**, **y** and **z** (flange frame, metres) for the same
three sources. The dataset observation uses the recorded `observation.state.ee_position[:3]`; the
dataset action and the robot reading are forward kinematics of `action.joint_target` / the recorded
robot joints (`action.ee_position` is a relative delta in this dataset and is ~0, so it cannot be used).

The final seven panels show joint **torque** [Nm]: the dataset's recorded `observation.state.joint_effort`
(blue), the torque the real controller induced (`FrankaState.tau_J_d`, red — the `Kp*(q_d - q) - Kd*dq`
PD law after the `torque_limit` clamp), and the torque the real robot measured (`FrankaState.tau_J`,
green). Grey dotted lines are the dataset's `current_effort_limit`. Note `tau_J_d` is gravity-free
while `tau_J` and the sim's `joint_effort` are not, so a constant per-joint offset between red and the
other two is expected — compare shape and saturation, not absolute level.

The dataset's final frame is dropped everywhere: its `action.joint_target` is all zeros (a bogus
trailing action) and the replay never executes it.

In [ ]:
assert "q_robot" in dir(), "run the robot execution cell first so q_robot is defined"
assert "q_sent" in dir(), "re-run the robot execution cell so q_sent / gripper_sent exist"
assert "tau_induced" in dir(), "re-run the robot execution cell so tau_induced / tau_measured exist"

# The dataset's last frame carries a bogus action (`action.joint_target` is all
# zeros there) and the replay never executes it (the cell above iterates over
# `actions[:-1]`), so drop it from every dataset trace.
T_dataset = min(len(q_obs), len(q_target), len(gripper_obs), len(gripper_action)) - 1
T_robot = min(len(q_robot), len(gripper_robot))
t_cmp = t[:T_dataset]
# Index robot samples by their intended fixed-rate schedule, rather than wall-clock
# time: changing PD gains can block briefly between two env steps.
t_robot = np.arange(T_robot) / FPS
q_obs_c = q_obs[:T_dataset]
q_target_c = q_target[:T_dataset]
q_robot_c = q_robot[:T_robot]
gripper_obs_c = gripper_obs[:T_dataset]
gripper_action_c = gripper_action[:T_dataset]
gripper_robot_c = gripper_robot[:T_robot]
# Actually-sent targets are recorded once per executed step, so they share the
# robot's ideal-FPS clock.
T_sent = min(len(q_sent), len(gripper_sent))
t_sent = np.arange(T_sent) / FPS
q_sent_c = q_sent[:T_sent]
gripper_sent_c = gripper_sent[:T_sent]

# Joint torques: dataset (sim) vs. the real controller's commanded and measured values.
T_tau = min(len(tau_induced), len(tau_measured))
t_tau = np.arange(T_tau) / FPS
tau_induced_c = tau_induced[:T_tau]
tau_measured_c = tau_measured[:T_tau]
tau_dataset_c = stack_col(ep, "observation.state.joint_effort")[:T_dataset]
# Recorded per-joint effort limit (constant over the episode); first 7 = arm.
effort_limit = stack_col(ep, "observation.state.current_effort_limit")[0, :7]

# End-effector x/y/z (flange frame, metres). The dataset stores the measured
# flange pose in `observation.state.ee_position`; for the commanded action and for
# the robot replay we only have joints, so both go through the same rcs forward
# kinematics (no TCP offset) to stay comparable with the recorded value.
import rcs
from matplotlib.ticker import AutoMinorLocator, MaxNLocator
from rcs import common as rcs_common

_fr3 = rcs.ROBOTS[rcs_common.RobotType.FR3]
_pin = rcs_common.Pin(_fr3.mjcf_model_path, _fr3.attachment_site, False)


def fk_xyz(q: np.ndarray) -> np.ndarray:
    """Flange position [m], (T, 3), for a (T, 7) stack of joint configurations."""
    if len(q) == 0:
        return np.empty((0, 3))
    return np.array([_pin.forward(np.asarray(qi, dtype=np.float64)).translation() for qi in q])


EE_AXES = ("x", "y", "z")
ee_obs_c = stack_col(ep, "observation.state.ee_position")[:T_dataset, :3]
ee_action_c = fk_xyz(q_target_c)
ee_robot_c = fk_xyz(q_robot_c)
ee_sent_c = fk_xyz(q_sent_c)

if T_robot:
    T_error = min(T_dataset, T_robot)
    err = np.abs(q_robot_c[:T_error] - q_target_c[:T_error])
    print(f"tracking error |robot - action|: mean={err.mean():.4f} rad, max={err.max():.4f} rad")
    err_ds = np.abs(q_robot_c[:T_error] - q_obs_c[:T_error])
    print(f"|robot - dataset obs|: mean={err_ds.mean():.4f} rad, max={err_ds.max():.4f} rad")
    err_ee = np.abs(ee_robot_c[:T_error] - ee_obs_c[:T_error])
    for _i, _ax_name in enumerate(EE_AXES):
        print(
            f"ee {_ax_name} |robot - dataset obs|: "
            f"mean={err_ee[:, _i].mean():.4f} m, max={err_ee[:, _i].max():.4f} m"
        )
    _dist = np.linalg.norm(ee_robot_c[:T_error] - ee_obs_c[:T_error], axis=1)
    print(f"ee position error (3D): mean={_dist.mean():.4f} m, max={_dist.max():.4f} m")

if T_tau and np.isfinite(tau_induced_c).any():
    T_tau_err = min(T_dataset, T_tau)
    _dtau = np.abs(tau_measured_c[:T_tau_err] - tau_dataset_c[:T_tau_err])
    print(
        f"|robot tau_J - dataset joint_effort|: mean={np.nanmean(_dtau):.2f} Nm, "
        f"max={np.nanmax(_dtau):.2f} Nm  (gravity included in both)"
    )
    _sat = np.abs(tau_induced_c) >= (effort_limit - 1e-6)
    print(
        "steps at the effort limit (induced torque clamped), per joint: "
        f"{np.nansum(_sat, axis=0).astype(int)} of {T_tau}"
    )
    print(
        "induced tau_J_d per joint: min="
        f"{np.round(np.nanmin(tau_induced_c, axis=0), 2)} max={np.round(np.nanmax(tau_induced_c, axis=0), 2)} Nm"
    )
elif T_tau:
    print("tau_induced / tau_measured are all NaN - re-run the robot cell and check its WARNING line")

if T_sent:
    T_sent_err = min(T_dataset, T_sent)
    err_cmd = np.abs(q_sent_c[:T_sent_err] - q_target_c[:T_sent_err])
    print(
        f"|sent - dataset action|: mean={err_cmd.mean():.4f} rad, max={err_cmd.max():.4f} rad"
        "  (non-zero => LimitedAbsoluteAction clipped the target)"
    )
    T_sent_rob = min(T_robot, T_sent)
    err_track = np.abs(q_robot_c[:T_sent_rob] - q_sent_c[:T_sent_rob])
    print(f"|robot - sent|: mean={err_track.mean():.4f} rad, max={err_track.max():.4f} rad")
else:
    print("No robot observations were recorded; showing dataset traces only.")

replay_status = "complete replay"
if "replay_error" in dir() and replay_error is not None:
    replay_status = f"replay stopped after {T_robot} samples ({type(replay_error).__name__})"

fig, axes = plt.subplots(9, 2, figsize=(13, 27), sharex=True)
axes = axes.ravel()
for j in range(7):
    ax = axes[j]
    ax.plot(t_cmp, q_obs_c[:, j], label="dataset obs", color="tab:blue", lw=1.5)
    ax.plot(t_cmp, q_target_c[:, j], label="dataset action", color="tab:orange", lw=1.0, ls="--")
    ax.plot(t_sent, q_sent_c[:, j], label="sent to robot (post-clip)", color="tab:red", lw=1.0, ls="-.")
    ax.plot(t_robot, q_robot_c[:, j], label="robot reading (ideal FPS)", color="tab:green", lw=1.2)
    ax.axhline(Q_HOME[j], color="grey", ls=":", lw=1.5, label="q_home")
    ax.set_title(f"joint_{j}")
    ax.set_ylabel("rad")
    ax.grid(alpha=0.3)
    if j == 0:
        ax.legend(loc="best", fontsize=8)

ax = axes[7]
ax.plot(t_cmp, gripper_obs_c, label="dataset obs", color="tab:blue", lw=1.5)
ax.plot(t_cmp, gripper_action_c, label="dataset action", color="tab:orange", lw=1.0, ls="--")
ax.plot(t_sent, gripper_sent_c, label="sent to robot (inverted)", color="tab:red", lw=1.0, ls="-.")
ax.plot(t_robot, gripper_robot_c, label="robot reading (ideal FPS)", color="tab:green", lw=1.2)
ax.set_title("gripper")
ax.grid(alpha=0.3)
ax.legend(loc="best", fontsize=8)

for k, ax_name in enumerate(EE_AXES):
    ax = axes[8 + k]
    ax.plot(t_cmp, ee_obs_c[:, k], label="dataset obs", color="tab:blue", lw=1.5)
    ax.plot(t_cmp, ee_action_c[:, k], label="dataset action (FK)", color="tab:orange", lw=1.0, ls="--")
    ax.plot(t_sent, ee_sent_c[:, k], label="sent to robot (FK, post-clip)", color="tab:red", lw=1.0, ls="-.")
    ax.plot(t_robot, ee_robot_c[:, k], label="robot reading (FK, ideal FPS)", color="tab:green", lw=1.2)
    ax.set_title(f"ee {ax_name} (flange)")
    ax.set_ylabel("m")
    if k == 0:
        ax.legend(loc="best", fontsize=8)

# tau_J_d is gravity-free and driven only by kp/kd, so it is typically one to two
# orders of magnitude smaller than tau_J (which carries the gravity load). Sharing
# one y-axis flattens it onto the zero line, so the induced torque gets its own
# right-hand axis, drawn in red and symmetric about zero.
tau_twin_axes = []
for j in range(7):
    ax = axes[11 + j]
    ax.plot(t_cmp, tau_dataset_c[:, j], label="dataset joint_effort", color="tab:blue", lw=1.5)
    ax.plot(t_tau, tau_measured_c[:, j], label="recorded (tau_J)", color="tab:green", lw=1.2)
    for sign in (1, -1):
        ax.axhline(sign * effort_limit[j], color="grey", ls=":", lw=1.5,
                   label="effort limit" if sign == 1 else None)
    ax.set_title(f"torque joint_{j}")
    ax.set_ylabel("Nm")

    ax_t = ax.twinx()
    tau_twin_axes.append(ax_t)
    ax_t.plot(
        t_tau, tau_induced_c[:, j],
        label="induced (tau_J_d, no gravity, right axis)", color="tab:red", lw=1.2, ls="-.",
    )
    _col = tau_induced_c[:, j]
    _amp = float(np.nanmax(np.abs(_col))) if np.isfinite(_col).any() else 0.0
    ax_t.set_ylim(-1.15 * _amp, 1.15 * _amp) if _amp > 0 else ax_t.set_ylim(-1.0, 1.0)
    ax_t.set_ylabel("Nm (induced)", color="tab:red", fontsize=8)
    ax_t.tick_params(axis="y", colors="tab:red", labelsize=8)
    ax_t.grid(False)
    if j == 0:
        _h1, _l1 = ax.get_legend_handles_labels()
        _h2, _l2 = ax_t.get_legend_handles_labels()
        ax.legend(_h1 + _h2, _l1 + _l2, loc="best", fontsize=8)

_visible_axes = [a for a in axes if a.get_visible()]

# Denser gridlines: more major ticks plus a minor grid, so values can be read
# off directly instead of eyeballed between two far-apart lines.
for ax in _visible_axes:
    ax.xaxis.set_major_locator(MaxNLocator(nbins=12, steps=[1, 2, 2.5, 5, 10]))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=8, steps=[1, 2, 2.5, 5, 10]))
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.grid(which="major", alpha=0.4, lw=0.8)
    ax.grid(which="minor", alpha=0.18, lw=0.5)
    ax.tick_params(labelsize=8)

for ax in (axes[16], axes[17]):
    ax.set_xlabel("time [s]")
fig.suptitle(
    f"Episode {EPISODE}: dataset vs. robot ({ROBOT_INSTANCE}; {replay_status})",
    y=1.0,
)
fig.tight_layout()
plt.show()

# Recorded ZED camera frames from the replay: a strip of frames per camera,
# sampled evenly across the replay, shown alongside the joint comparison above.
_cam_frames = camera_frames if "camera_frames" in dir() else {}
if _cam_frames:
    N_STRIP = 6
    cam_names = list(_cam_frames.keys())
    fig_cam, axes_cam = plt.subplots(
        len(cam_names), N_STRIP, figsize=(3 * N_STRIP, 2.6 * len(cam_names)), squeeze=False
    )
    for row, cam in enumerate(cam_names):
        frames = _cam_frames[cam]
        strip_idx = np.linspace(0, len(frames) - 1, N_STRIP).astype(int)
        for col, fi in enumerate(strip_idx):
            ax = axes_cam[row, col]
            ax.axis("off")
            img = np.asarray(frames[fi])
            if img.ndim == 3 and img.shape[2] == 4:  # RGBA -> RGB
                img = img[:, :, :3]
            ax.imshow(img)
            if row == 0:
                ax.set_title(f"t = {fi / FPS:.1f} s", fontsize=10)
            if col == 0:
                ax.text(-0.05, 0.5, cam, transform=ax.transAxes, rotation=90,
                        va="center", ha="right", fontsize=11)
    fig_cam.suptitle(f"Episode {EPISODE}: recorded ZED camera frames ({replay_status})", y=1.0)
    fig_cam.tight_layout()
    plt.show()
else:
    print("No ZED camera frames were recorded (set ZED_CAMERA_DICT and run the execution cell).")


## Is the leftover z error gravity or lag?

The payload knob can only add a *constant* force, so it can only shift the whole
trajectory: over-declare and the arm rides high (lifts further, refuses to come
back down), under-declare and it sags everywhere. The PD law's velocity lag
(`tau = Kp*(q_des - q) - Kd*dq`, no feedforward) is the opposite shape --
proportional to speed, so it pulls *both* the top and the bottom of a move toward
the middle and leaves the resting points alone.

Fitting `e_z = bias + slope * v_z` separates them, so you can tell whether the
next hardware run should change `M_CAMERA` or stop trying to.

In [ ]:
# Soft-mode gains from set_pd_mode(0.0) -- keep in sync if you retune them.
_KP_SOFT, _KD_SOFT = 40.0, 13.0

T_z = min(len(ee_robot_c), len(ee_sent_c))
if T_z > 3:
    z_robot = ee_robot_c[:T_z, 2]
    z_sent = ee_sent_c[:T_z, 2]
    e_z = z_robot - z_sent
    v_z = np.gradient(z_sent) * FPS
    # z_robot(t) ~= z_sent(t - tau) ~= z_sent(t) - tau * v_z  =>  slope = -tau
    bias, slope = np.linalg.lstsq(np.column_stack([np.ones(T_z), v_z]), e_z, rcond=None)[0]
    still = np.abs(v_z) < 0.02  # m/s

    print(f"z error decomposition over {T_z} samples")
    print(f"  bias                     = {bias * 1e3:+7.1f} mm    <- gravity / declared payload")
    print(f"  lag                      = {-slope * 1e3:7.1f} ms    <- velocity-proportional")
    print(f"  Kd/Kp (predicted lag)    = {_KD_SOFT / _KP_SOFT * 1e3:7.1f} ms")
    if still.any():
        print(
            f"  mean e_z while nearly still = {e_z[still].mean() * 1e3:+.1f} mm "
            f"over {int(still.sum())} samples   <- the only part payload can fix"
        )
    print(
        f"  peak-to-peak z: commanded {np.ptp(z_sent) * 1e3:.0f} mm, "
        f"robot {np.ptp(z_robot) * 1e3:.0f} mm "
        f"({np.ptp(z_robot) / max(np.ptp(z_sent), 1e-9):.0%} of commanded)"
    )
    print(
        "\nbias > 0: over-compensating, lower M_CAMERA.  bias < 0: raise it.\n"
        "Once |bias| is small, whatever amplitude is still missing is lag, and\n"
        "adding mass to chase it only re-introduces the bias."
    )


## Render recorded ZED frames to mp4 (ffmpeg)

Encodes the ZED RGB frames captured during the replay (`camera_frames`) into an mp4 per
camera using `ffmpeg` (raw RGB frames piped over stdin, `libx264` / `yuv420p`), then embeds
the players inline. Requires `ffmpeg` on `PATH` and the execution cell to have been run.


In [ ]:
import shutil
import subprocess
from pathlib import Path

from IPython.display import HTML, display

assert shutil.which("ffmpeg") is not None, "ffmpeg not found on PATH"

VIDEO_DIR = Path("replay_videos_switch")
VIDEO_DIR.mkdir(exist_ok=True)


def render_mp4_ffmpeg(frames, path: Path, fps: float) -> Path | None:
    """Encode a list of RGB(A) uint8 frames to an mp4 via ffmpeg (rawvideo pipe)."""
    frames = [np.asarray(f) for f in frames]
    if not frames:
        return None
    h, w = frames[0].shape[:2]
    cmd = [
        "ffmpeg", "-y",
        "-f", "rawvideo",
        "-pix_fmt", "rgb24",
        "-s", f"{w}x{h}",
        "-r", f"{fps}",
        "-i", "-",
        "-an",
        "-vcodec", "libx264",
        "-pix_fmt", "yuv420p",
        str(path),
    ]
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
    assert proc.stdin is not None
    for frame in frames:
        if frame.ndim == 3 and frame.shape[2] == 4:  # RGBA -> RGB
            frame = frame[:, :, :3]
        if frame.dtype != np.uint8:
            frame = frame.astype(np.uint8)
        proc.stdin.write(np.ascontiguousarray(frame).tobytes())
    proc.stdin.close()
    if proc.wait() != 0:
        raise RuntimeError((proc.stderr.read() if proc.stderr else b"").decode())
    return path


_cam_frames = camera_frames if "camera_frames" in dir() else {}
if _cam_frames:
    tags = []
    for cam, frames in _cam_frames.items():
        out = VIDEO_DIR / f"episode{EPISODE}_{cam}.mp4"
        render_mp4_ffmpeg(frames, out, FPS)
        print(f"wrote {out} ({len(frames)} frames)")
        b64 = base64.b64encode(out.read_bytes()).decode()
        tags.append(
            f"<div style='width:48%'><b>{cam}</b><br>"
            f"<video controls loop muted style='width:100%'>"
            f"<source src='data:video/mp4;base64,{b64}' type='video/mp4'></video></div>"
        )
    display(HTML(f"<div style='display:flex;flex-wrap:wrap;gap:2%'>{''.join(tags)}</div>"))
else:
    print("No ZED camera frames were recorded (run the execution cell with ZED cameras).")
